# Lecture 9: Shooting the swing-up

**The challenge.** Lectures 6 to 8 built a fast, accurate forward map, from
an initial costate to an extremal, with its exact derivatives. But the
boundary value problem of Lecture 3 does not give us the initial costate: it
gives half the boundary conditions at each end. Something has to turn "these
conditions at $t_f$ are missed by this much" into "change the initial
costate by that much", reliably, for a problem whose dynamics stretch errors
exponentially. That something is a root finder wrapped around the forward
map: **shooting**.

**In this lecture.** We solve the boundary value problem derived in
Lecture 3 for Kelly's cart-pole swing-up. We start with the naive approach,
guessing the initial costate, and see why it fails. Then we try the bluntest
remedy, random multistart, and compare what it finds with the published
solution, computed by a direct (collocation) method. The comparison shows
that the necessary conditions have *more than one* solution here, so "the"
optimal swing-up has a competitor. We finish by checking the results
independently of the package.

**Prerequisites.** Lecture 3 (the canonical equations of the cart-pole) and
Lectures 6 to 8 (the kernels, events and the STM).

**After this lecture you can**

- set up and solve a single-shooting problem with the package;
- explain why indirect methods need good costate guesses, and get them from
  random search;
- tell a converged extremal from a verified optimum, and check a solution
  independently.

Background: Stanford AA203 lecture 5 ("indirect methods: practical aspects"),
MIT 16.323 lecture notes 6 and 8, and Kelly (2017), the source of the
problem and of the direct method we compare with in section 4.

## A bit of history: aiming a cannon at a boundary condition

The name is literal. To hit a target with a cannon, fire, see where the ball
lands, adjust the elevation, fire again. A two-point boundary value problem
is solved the same way: "fire" an initial value problem from a guessed
initial condition, measure the miss at the far end, and correct the guess.
Each correction is a step of Newton's method for the miss as a function of
the guess, a method Newton sketched around 1669 for polynomials and Joseph
Raphson published in its iterative form in 1690.

Shooting became a practical tool with digital computers. Morrison, Riley and
Zancanaro described *multiple* shooting in 1962, splitting the interval to
tame the growth of errors, and Herbert Keller's 1968 book *Numerical Methods
for Two-Point Boundary-Value Problems* gave the analysis. At the Technical
University of Munich, Roland Bulirsch, Josef Stoer and their students turned
multiple shooting into the workhorse of the indirect method for aerospace
problems in the 1970s; Hans Joachim Oberle's code BNDSCO, from that school,
was a standard for decades. The showcase example of Stoer and Bulirsch's
numerical analysis textbook is the optimal re-entry of an Apollo capsule.

The root finders the package uses come from MINPACK, the Fortran library Jorge
Moré, Burton Garbow and Kenneth Hillstrom wrote at Argonne National
Laboratory in 1980: Powell's 1970 *hybrid* method, which blends Newton steps
with steepest descent inside a trust region, and Levenberg–Marquardt. SciPy
still calls the same Fortran, translated to C.

## 1. The problem, posed for the package

The same plant and boundary conditions as Lecture 3, with Kelly's cost
$J = \int_0^T u^2\,dt$ and Kelly's force limit $|u| \le 20$ N. The control
law is then the clipped one from Lecture 4.

The limit turns out to be inactive at the solution (we check that in
section 5), but it still matters, for the solver rather than the answer. With
no bound, a wild costate guess commands an enormous force, the cart-pole
spins up violently, and the integrator crawls through it. One failed start
can then take seconds instead of milliseconds. With the bound, every guess
integrates quickly. Kelly also limits the track to $|q_1| \le 2$ m. We leave
that out and check it at the end. (Lecture 13 brings the track ends back as a
path constraint.)

The constants are SymPy symbols, so they become **runtime parameters** of the
compiled kernels: one compilation serves every value.

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import sympy as sp

from auto_optimal_control.build import build_kernels
from auto_optimal_control.pontryagin import derive
from auto_optimal_control.problem import Box, OptimalControlProblem
from auto_optimal_control.shooting import ShootingError, SingleShooting

BLUE, ORANGE, GRAY = "#2a78d6", "#eb6834", "#8a8a85"

t = sp.Symbol("t", real=True)
q1, q2, w1, w2, u = sp.symbols("q_1 q_2 qd_1 qd_2 u", real=True)
m1, m2, l, g, u_max, d, T = sp.symbols("m_1 m_2 l g u_max d T", positive=True)

den = m1 + m2 * (1 - sp.cos(q2) ** 2)
q1dd = (
    l * m2 * sp.sin(q2) * w2**2 + u + m2 * g * sp.cos(q2) * sp.sin(q2)
) / den
q2dd = -(
    l * m2 * sp.cos(q2) * sp.sin(q2) * w2**2
    + u * sp.cos(q2)
    + (m1 + m2) * g * sp.sin(q2)
) / (l * den)

problem = OptimalControlProblem(
    name="cart_pole",
    time=t,
    states=[q1, q2, w1, w2],
    controls=[u],
    params=[m1, m2, l, g, u_max, d, T],
    dynamics=[w1, w2, q1dd, q2dd],
    running_cost=u**2,
    initial_state=[0, 0, 0, 0],
    final_state={q1: d, q2: sp.pi, w1: 0, w2: 0},
    final_time=T,
    control_sets=[Box(u, -u_max, u_max)],
)

start = time.perf_counter()
kernels = build_kernels(derive(problem))
print(f"derived and compiled in {time.perf_counter() - start:.1f} s")
p = [1.0, 0.3, 0.5, 9.81, 20.0, 1.0, 2.0]  # m1, m2, l, g, u_max, d, T
kernels.param_names

`SingleShooting` wraps the kernels in the residual $R(v)$ of Lecture 8 and its
exact Jacobian, where $v$ collects the unknowns: the initial costate
$\lambda_0$ and the multipliers $\nu$ of the terminal conditions. `solve` hands
both to SciPy's Powell hybrid method. `guess_multipliers` takes a costate
guess and fills in the $\nu$ that best fit it. The transversality conditions
are linear in $\nu$, so only $\lambda_0$ has to be guessed.

In [ ]:
shoot = SingleShooting(kernels, p)
shoot

## 2. A first guess

With no information, the natural guess is $\lambda_0 = 0$: no idea which way
to push. Try it.

In [ ]:
sol0 = shoot.solve(shoot.guess_multipliers(np.zeros(4)))
J0 = kernels.propagate(0.0, sol0.z0, p[-1], p, cost=True)["cost"]
print(f"residual norm {sol0.residual_norm:.1e}, success: {sol0.success}")
print(f"J = {J0:.6f}")
print("lambda0 =", np.round(sol0.lam0, 4))

It converged, to a swing-up with $J \approx 57.94$. That is luck, not
design. At $\lambda_0 = 0$ the costate stays zero, the force is zero, and the
pendulum just hangs: the first Newton step has nothing to go on but the
Jacobian there, and it happens to point somewhere useful. Nothing guarantees
that for another problem, or even for this one with other parameters.

The underlying trouble is **sensitivity**. The cell below integrates from
a few random costates and reports how large the state transition matrix gets
by $t = T$, together with the condition number of the shooting Jacobian
$\partial x(T) / \partial \lambda_0$. A small change in $\lambda_0$ makes a
large change at the far end, and the Jacobian is far from well conditioned.

In [ ]:
rng = np.random.default_rng(0)
for _ in range(4):
    lam0 = rng.uniform(-30, 30, 4)
    r = kernels.propagate(0.0, kernels.z0(lam0, p), p[-1], p, stm=True)
    Phi = r["stm"]
    print(
        f"|Phi(T)| = {np.linalg.norm(Phi):9.3g}   "
        f"cond(dx(T)/dlam0) = {np.linalg.cond(Phi[:4, 4:]):9.3g}"
    )

## 3. Remedy one: random multistart

The bluntest remedy is to try many guesses. Each `solve` is cheap, because the
kernels are compiled C integrated by a compiled integrator, so we can afford
hundreds. We draw $\lambda_0$ uniformly from a box, solve from each draw, and
keep every converged extremal, grouped by its cost.

In [ ]:
def solve_from(lam0):
    sol = shoot.solve(shoot.guess_multipliers(lam0))
    if not sol.success:
        return None
    cost = kernels.propagate(0.0, sol.z0, p[-1], p, cost=True)["cost"]
    return sol, cost


rng = np.random.default_rng(0)
found = {}
start = time.perf_counter()
n_tries, n_converged = 200, 0
for _ in range(n_tries):
    try:
        out = solve_from(rng.uniform(-60, 60, 4))
    except (ShootingError, RuntimeError, ValueError, np.linalg.LinAlgError):
        continue
    if out is not None:
        sol, J = out
        n_converged += 1
        key = round(J, 6)
        found.setdefault(key, [sol, 0])[1] += 1
elapsed = time.perf_counter() - start
print(f"{n_tries} starts in {elapsed:.1f} s, {n_converged} converged")
for J in sorted(found):
    print(f"J = {J:10.6f}   found {found[J][1]:3d} times")

Only about one start in eight converges, and every one that does lands on the
same extremal as the zero guess, $J \approx 57.94$. That says something about
the basins of attraction of Newton's method in costate space: they are small
and irregular, and a uniform box mostly misses them. It does *not* say that
this extremal is the only one. Section 4 shows another.

### The shooting system in full

`SingleShooting` is a thin, explicit layer, and worth knowing in detail,
because every later solver is built from it.

- **Unknowns** `v`, in order: the free initial states, the initial costates,
  the terminal multipliers $\nu$, and $t_f$ when it is free. `pack` assembles
  `v` and `unpack` takes it apart; `n_unknowns` is its length.
- **Residual** $R(v)$: the boundary conditions of Lecture 2, with the fixed
  initial states eliminated. `residual(v)` evaluates it, and
  `residual_jacobian(v)` returns it with its exact Jacobian (Lecture 8).
  Both need only NumPy, so any root finder or sampler can drive them.
- **`guess_multipliers(lam0, tf)`** fills in $\nu$ by least squares, since the
  transversality conditions are linear in it: only the costate has to be
  guessed.
- **`solve(guess, method="hybr" or "lm", tol=1e-10)`** calls MINPACK through
  SciPy and returns a `ShootingSolution`: `v`, `x0`, `lam0`, `nu`, `tf`,
  `residual_norm`, `success`, `nfev` and `max_violation` (Lecture 13).
  A guess whose propagation fails (Lecture 7's statuses) is reported to the
  root finder as a large residual, so a bad guess fails the solve instead of
  raising.
- **`multistart(sample, n_tries, seed)`** solves from random guesses
  `sample(rng)` until one succeeds, and returns it with the number of tries.
- **`trajectory(v)`** samples the extremal; `constraint_violation(v)` checks
  path constraints.

Here they are on a problem with a free final time, Zermelo's boat from
Lecture 4, where $t_f$ is an unknown and $H(t_f) = 0$ the extra residual.

In [ ]:
from auto_optimal_control import examples

kz = build_kernels(derive(examples.zermelo()))
zshoot = SingleShooting(kz, [1.0, 1.0, 1.0, 1.0])       # V, k, x_T, y_T
print(zshoot, "\nunknowns:", zshoot.n_unknowns)
zsol, tries = zshoot.multistart(
    lambda rng: zshoot.guess_multipliers(rng.uniform(-2, 2, 2), tf=rng.uniform(0.5, 3.0)),
    n_tries=50, seed=0)
print(f"converged after {tries} draw(s): t_f = {zsol.tf:.10f}, |R| = {zsol.residual_norm:.1e}, "
      f"{zsol.nfev} evaluations")
x0_, lam0_, nu_, tf_ = zshoot.unpack(zsol.v)
print("lam0 =", lam0_, " nu =", nu_)
same = zshoot.solve(zsol.v + 1e-3, method="lm")
print(f"Levenberg-Marquardt from a nearby guess: t_f = {same.tf:.10f}")

## 4. A comparison: the direct method

A **direct** method discretizes first and optimizes second. It turns the
problem into a finite nonlinear program over the states and controls at grid
points, and gives that to a general-purpose optimizer. It is robust to poor
guesses (a straight line will do) but only as accurate as its grid. The
published solution of this problem comes from one. We run Kelly's here
*only to compare answers*: nothing in the package, and nothing in this
course, starts from a direct solution.

Kelly's method is Hermite–Simpson collocation on 40 intervals, with SciPy's
SLSQP. The *defects* are the collocation equations, the requirement that
each interval's cubic interpolant satisfy the dynamics at its ends and its
midpoint. It needs a NumPy version of the plant.

In [ ]:
def plant(x, u, p):
    m1, m2, l, g = p[:4]
    q2, w2 = x[1], x[3]
    den = m1 + m2 * (1 - np.cos(q2) ** 2)
    return np.array(
        [
            x[2],
            w2,
            (
                l * m2 * np.sin(q2) * w2**2
                + u
                + m2 * g * np.cos(q2) * np.sin(q2)
            )
            / den,
            -(
                l * m2 * np.cos(q2) * np.sin(q2) * w2**2
                + u * np.cos(q2)
                + (m1 + m2) * g * np.sin(q2)
            )
            / (l * den),
        ]
    )


def hermite_simpson(p, N=40, x_max=2.0):
    from scipy.optimize import minimize

    d, T = p[5], p[6]
    h = T / N

    def split(w):
        return w[: 4 * (N + 1)].reshape(N + 1, 4), w[4 * (N + 1) :]

    def defects(w):
        X, U = split(w)
        out = []
        for i in range(N):
            fk, fk1 = plant(X[i], U[i], p), plant(X[i + 1], U[i + 1], p)
            xm = 0.5 * (X[i] + X[i + 1]) + h / 8 * (fk - fk1)
            um = 0.5 * (U[i] + U[i + 1])
            out.append(
                X[i + 1] - X[i] - h / 6 * (fk + 4 * plant(xm, um, p) + fk1)
            )
        out += [X[0], X[-1] - [d, np.pi, 0, 0]]
        return np.concatenate(out)

    def cost(w):
        _, U = split(w)
        um = 0.5 * (U[:-1] + U[1:])
        return h / 6 * np.sum(U[:-1] ** 2 + 4 * um**2 + U[1:] ** 2)

    tg = np.linspace(0, T, N + 1)
    # Straight-line guess: cart and pole move uniformly, at zero velocity.
    line = np.stack([d * tg / T, np.pi * tg / T, 0 * tg, 0 * tg], axis=1)
    w0 = np.concatenate([line.ravel(), np.zeros(N + 1)])
    bounds = [(-x_max, x_max), (None, None), (None, None), (None, None)] * (
        N + 1
    ) + [(-p[4], p[4])] * (N + 1)
    res = minimize(
        cost,
        w0,
        method="SLSQP",
        constraints={"type": "eq", "fun": defects},
        bounds=bounds,
        options={"maxiter": 500, "ftol": 1e-12},
    )
    X, U = split(res.x)
    return tg, X, U, cost(res.x)


start = time.perf_counter()
tg, X, U, J_direct = hermite_simpson(p)
elapsed = time.perf_counter() - start
print(f"direct solution: J = {J_direct:.5f} in {elapsed:.1f} s")

### Two different answers

The direct method lands on Kelly's published cost. That is not the cost of
the extremal the random search found.

In [ ]:
J_best = min(found)
print(f"J direct, N = 40    = {J_direct:.6f}")
print("J published         = 58.80819   (Kelly's problem, high-order direct method)")
print(f"J cheapest extremal = {J_best:.6f}   (random search, integration tolerance)")

The direct solution matches the published cost to the accuracy of its
40-interval grid. But it is **not** the extremal that multistart found, and it
costs more. The necessary conditions have at least two solutions here, and
the one in the literature is the more expensive. Kelly's direct method
started from a straight line and converged to a local minimum; the random
search happened to find a better one. Neither method promises a global
optimum.

In [ ]:
from scipy.interpolate import CubicSpline

fig, axes = plt.subplots(1, 3, figsize=(11, 3.2), sharex=True)
tr = shoot.trajectory(found[J_best][0].v, n_points=401)
ts = np.linspace(0, p[-1], 401)
Xs, Us = CubicSpline(tg, X)(ts), CubicSpline(tg, U)(ts)
axes[0].plot(ts, Xs[:, 0], color=BLUE, lw=2, label=f"direct, J = {J_direct:.4f}")
axes[1].plot(ts, Xs[:, 1], color=BLUE, lw=2)
axes[2].plot(ts, Us, color=BLUE, lw=2)
axes[0].plot(tr["t"], tr["z"][:, 0], color=ORANGE, lw=2, label=f"extremal, J = {J_best:.4f}")
axes[1].plot(tr["t"], tr["z"][:, 1], color=ORANGE, lw=2)
axes[2].plot(tr["t"], tr["u"][:, 0], color=ORANGE, lw=2)
axes[0].plot(tg, X[:, 0], "o", color=GRAY, ms=3, label="direct grid, N = 40")
axes[1].plot(tg, X[:, 1], "o", color=GRAY, ms=3)
axes[2].plot(tg, U, "o", color=GRAY, ms=3)
titles = ["cart position $q_1$ (m)", "pole angle $q_2$ (rad)", "force $u$ (N)"]
for ax, label in zip(axes, titles):
    ax.set_title(label, fontsize=10, loc="left")
    ax.set_xlabel("t (s)")
    ax.grid(alpha=0.3)
axes[0].legend(frameon=False, fontsize=9)
fig.tight_layout()

The two swing-ups are genuinely different. The direct solution (blue)
pushes the cart forward at once, overshoots the target to 1.18 m and comes
back. The cheaper extremal (orange) first pulls the cart *backwards*, which
kicks the pole the other way, and never passes $q_1 = 1$.

## 5. Checking the answer

A converged shooting solve says $R(v) \approx 0$ in the package's own
equations. An independent check uses none of them: take the extremal's
control history $u(t)$, drive Kelly's equations of motion with SciPy's
integrator, and see whether the cart arrives at the target with the claimed
cost. The same check, applied to the direct solution's control, shows how
far its grid is from the equations of motion. We also check two properties the theory predicts:

- $H$ is constant along the extremal, since the problem is autonomous;
- the force limit is inactive ($|u| < 20$ N throughout, so the clipped law
  never clips), and the track limit $|q_1| \le 2$ m that we left out holds
  anyway.

In [ ]:
from scipy.integrate import solve_ivp
from scipy.interpolate import CubicSpline


def simulate(u_of_t):
    def f(s, y):
        us = float(u_of_t(s))
        return np.append(plant(y[:4], us, p), us**2)

    sim = solve_ivp(
        f, (0.0, p[-1]), np.zeros(5), method="DOP853", rtol=1e-12, atol=1e-12
    )
    return sim.y[4, -1], np.abs(sim.y[:4, -1] - [p[5], np.pi, 0, 0]).max()


def check(v, label):
    tr = shoot.trajectory(v, n_points=4001)
    u_of_t = CubicSpline(tr["t"], tr["u"][:, 0])
    J_sim, end_error = simulate(u_of_t)
    H_spread = np.ptp(tr["H"]) / abs(tr["H"].mean())
    u_peak = np.abs(tr["u"]).max()
    q1_peak = np.abs(tr["z"][:, 0]).max()
    print(f"{label}:")
    print(f"    simulated J = {J_sim:.8f}, end-state error {end_error:.1e}")
    print(f"    H varies by {H_spread:.0e} (relative)")
    print(f"    max|u| = {u_peak:.2f} N, max|q1| = {q1_peak:.2f} m")


check(found[J_best][0].v, "cheapest extremal")
J_sim, end_error = simulate(CubicSpline(tg, U))
print("direct solution (its control, splined):")
print(f"    simulated J = {J_sim:.8f}, end-state error {end_error:.1e}")

The extremal checks out. The simulated cost matches to eight figures, the
cart arrives to within $10^{-7}$ (the spline of $u(t)$ limits this), $H$ is
constant to better than $10^{-7}$ of its value, and it stays inside Kelly's
bounds. The force peaks well below 20 N: the limit is inactive here. The direct solution's control, simulated
the same way, misses the target badly. Its control is right only to the
accuracy of its grid, and played open loop, the swing-up amplifies that
error: the same sensitivity that makes shooting hard, seen from the other
side.

## Summary

- Single shooting solves the boundary value problem by root-finding on the
  initial costate. It converges fast and accurately from a good guess, but
  costates have no physical intuition and the map $\lambda_0 \mapsto x(T)$ is
  sensitive, so good guesses are the hard part.
- The bluntest remedy is many random starts, made affordable by compiled
  kernels. Lecture 10 adds continuation, and the package's
  `homotopy.solve` automates it.
- A direct method, run for comparison, converged elsewhere: to the published
  solution, which costs more than the extremal the search found.
- The necessary conditions can have several solutions. Ranking them needs the
  cost, and proving global optimality needs more than first-order
  conditions. Neither method guarantees it.

## Exercises

1. Change the box in section 3 from $[-60, 60]^4$ to $[-10, 10]^4$ and then
   $[-200, 200]^4$. How does the fraction of converged starts change? Does
   either box find the literature extremal? (Hint: look at its $\lambda_0$.)
2. Run `hermite_simpson` with $N = 10, 20, 40, 80$ and simulate each
   control as in section 5. Plot the end-state error against $N$ on log axes.
   What order of convergence do you see? (Hermite–Simpson is fourth order in
   the state; Kelly 2017, section 4.)
3. Seed SLSQP from the cheapest extremal's trajectory instead of a straight
   line. Does the direct method then find the cheaper solution?

## Further reading

- M. Kelly, "An Introduction to Trajectory Optimization: How to Do Your Own
  Direct Collocation", SIAM Review 59(4), 2017, sections 4 and 6.
- Stanford AA203, lecture 5: indirect methods in practice.
- J. T. Betts, *Practical Methods for Optimal Control Using Nonlinear
  Programming* (SIAM, 2010), chapter 4, on direct methods.
- MIT 16.323, lecture notes 8: properties of the optimal solution.